# VP Applied AI/ML — Payments Engineering Case

**Candidate:** ____  **Date:** ____

This notebook is your **single deliverable**. It is self-contained: it does not import from `src/` and needs only
`numpy`, `pandas` and `scikit-learn` plus the files in `data/` and `artifacts/`.

**Rules for the submission**

1. It must run top to bottom with *Kernel → Restart & Run All*. Unfinished stages are fine: leave their functions raising
   `NotImplementedError`; the notebook catches it and moves on.
2. Keep outputs in the saved notebook.
3. For each stage, fill in the assumptions / results / limitations cell and the short **AI assistant log**.
4. Stages 3–7 are released progressively by the interviewer. Paste each brief into its stage section.
5. Do not use network calls or paid APIs. English only.

## 0. Setup

In [ ]:
import os
import sys
from collections import Counter, deque
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

RANDOM_SEED = 7
EXTRACTION_DATE = pd.Timestamp("2026-10-05", tz="UTC")
LABEL_MATURITY_DAYS = 45


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Locate the case repository: PAYMENTS_CASE_ROOT if set, else the cwd or one of its parents."""
    if os.environ.get("PAYMENTS_CASE_ROOT"):
        return Path(os.environ["PAYMENTS_CASE_ROOT"]).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "transactions_live.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/transactions_live.csv above " + str(start))


ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("Repository root:", ROOT)

In [ ]:
def run_checks(checks) -> pd.DataFrame:
    """Run plain-assert check functions; unfinished stages report NOT IMPLEMENTED instead of crashing."""
    results = {}
    for check in checks:
        try:
            check()
            results[check.__name__] = "PASS"
        except NotImplementedError:
            results[check.__name__] = "NOT IMPLEMENTED"
        except AssertionError as exc:
            results[check.__name__] = f"FAIL {exc}".strip()
        except Exception as exc:  # surface bugs without stopping Run All
            results[check.__name__] = f"ERROR {type(exc).__name__}: {exc}"
    return pd.Series(results, name="status").to_frame()


def try_stage(label, fn, *args, **kwargs):
    """Call a stage function; return None if it is not implemented yet."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"[{label}] not implemented yet")
        return None

In [ ]:
transactions = pd.read_csv(DATA_DIR / "transactions_live.csv")
transactions["created_at"] = pd.to_datetime(transactions["created_at"], utc=True)
labels = pd.read_csv(DATA_DIR / "labels_live.csv")
labels["chargeback_reported_at"] = pd.to_datetime(labels["chargeback_reported_at"], utc=True)
merchants = pd.read_csv(DATA_DIR / "merchants.csv")
acquirers = pd.read_csv(DATA_DIR / "acquirer_reference.csv")
messages = pd.read_csv(DATA_DIR / "payment_exception_messages.csv")
messages["created_at"] = pd.to_datetime(messages["created_at"], utc=True)
embeddings = pd.read_csv(DATA_DIR / "payment_exception_embeddings.csv")

for name, frame in [("transactions", transactions), ("labels", labels), ("merchants", merchants),
                    ("acquirers", acquirers), ("messages", messages), ("embeddings", embeddings)]:
    print(f"{name:<13} {frame.shape}")

---
# Stage 1 — Data Audit and Decision-Point Semantics

You receive historical gateway events (`transactions`), merchant metadata (`merchants`), downstream outcomes (`labels`)
and routing references (`acquirers`). See `docs/DATA_DICTIONARY.md` for column availability.

**The model is asked to score a payment before an acquirer is selected.**

**Deliverable:** concise reproducible audit summary and your modeling target definition. Do not start modeling until your
feature-availability assumptions are explicit.

In [ ]:
def audit_data(transactions: pd.DataFrame, labels: pd.DataFrame) -> dict:
    """Return a useful, auditable overview of the raw data."""
    raise NotImplementedError("Stage 1")


audit = try_stage("Stage 1", audit_data, transactions, labels)
audit

**Stage 1 — notes**

- Target definition:
- Approach:
- Assumptions:
- Findings:
- Open questions:

**AI assistant log — Stage 1**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 2 — Real-Time Feature Engineering

Implement `build_online_features(transactions)`. For each **unique logical payment** produce:

- `customer_txn_count_10m`: number of earlier transactions from the customer within 10 minutes;
- `customer_amount_sum_1h`: previous customer amount total within 1 hour;
- `customer_unique_merchants_24h`: distinct earlier merchant IDs within 24 hours;
- `seconds_since_customer_prev_txn`: seconds since the most recent strictly earlier customer transaction (`NaN` if none);
- `device_seen_before`: `1` iff this customer's device appeared strictly earlier, otherwise `0`;
- `ip_country_changed_from_prev_txn`: `1` if IP country differs from the most recent earlier payment's IP country, otherwise `0` (`0` if none).

**Required semantics**

- Output has **one row per logical `transaction_id`** and retains pre-decision fields.
- The gateway export is unsorted and contains retry events.
- **Strictly before** means events with the exact same customer and timestamp **must not see each other**.
- Window semantics: `(current_timestamp - window, current_timestamp)` (exclusive on both ends). A payment exactly at the left boundary is excluded.
- Avoid quadratic scans; target approximately linearithmic complexity using time-ordered state.
- Use event timestamps in UTC; explain behavior for out-of-order arrival in production.

**Follow-up:** What changes for 50,000 requests/s and events arriving 90 seconds late?

In [ ]:
FEATURE_COLUMNS = [
    "customer_txn_count_10m",
    "customer_amount_sum_1h",
    "customer_unique_merchants_24h",
    "seconds_since_customer_prev_txn",
    "device_seen_before",
    "ip_country_changed_from_prev_txn",
]


def build_online_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Strictly earlier, deduplicated, time-windowed customer features."""
    raise NotImplementedError("Stage 2")

### Stage 2 — contract checks (provided)
These mirror the public tests. Do not edit them.

In [ ]:
def _sample_events() -> pd.DataFrame:
    rows = [
        ["E1", "T1", "2026-06-01T10:00:00Z", "C1", "M1", "D1", 10, "GB"],
        ["E2", "T2", "2026-06-01T10:00:00Z", "C1", "M2", "D1", 20, "FR"],
        ["E3", "T3", "2026-06-01T10:05:00Z", "C1", "M3", "D1", 30, "FR"],
        ["E4", "T4", "2026-06-01T11:05:00Z", "C1", "M1", "D2", 40, "US"],
        ["E5", "T5", "2026-06-01T10:05:00Z", "C2", "M2", "D9", 9, "GB"],
        ["E_RETRY", "T1", "2026-06-01T10:00:00Z", "C1", "M1", "D1", 10, "GB"],
    ]
    frame = pd.DataFrame(rows, columns=["event_id", "transaction_id", "created_at", "customer_id",
                                        "merchant_id", "device_id", "amount", "ip_country"])
    frame["created_at"] = pd.to_datetime(frame["created_at"], utc=True)
    return frame


def check_strictly_prior_same_timestamp_and_retries():
    f = build_online_features(_sample_events()).set_index("transaction_id")
    assert len(f) == 5, f"expected 5 logical payments, got {len(f)}"
    assert f.loc["T1", "customer_txn_count_10m"] == 0
    assert f.loc["T2", "customer_txn_count_10m"] == 0
    assert f.loc["T3", "customer_txn_count_10m"] == 2
    assert np.isclose(f.loc["T3", "customer_amount_sum_1h"], 30.0)
    assert f.loc["T3", "customer_unique_merchants_24h"] == 2
    assert np.isclose(f.loc["T3", "seconds_since_customer_prev_txn"], 300)
    assert f.loc["T3", "device_seen_before"] == 1
    assert f.loc["T5", "customer_txn_count_10m"] == 0


def check_exact_window_boundary_and_first_observation():
    f = build_online_features(_sample_events()).set_index("transaction_id")
    assert f.loc["T4", "customer_txn_count_10m"] == 0
    assert np.isclose(f.loc["T4", "customer_amount_sum_1h"], 0.0)
    assert f.loc["T4", "customer_unique_merchants_24h"] == 3
    assert f.loc["T4", "device_seen_before"] == 0
    assert np.isnan(f.loc["T1", "seconds_since_customer_prev_txn"])
    assert f.loc["T1", "ip_country_changed_from_prev_txn"] == 0
    assert f.loc["T4", "ip_country_changed_from_prev_txn"] == 1


def check_shuffled_input_does_not_change_results():
    df = _sample_events()
    a = build_online_features(df).set_index("transaction_id").sort_index()
    b = build_online_features(df.sample(frac=1, random_state=144)).set_index("transaction_id").sort_index()
    pd.testing.assert_frame_equal(a[FEATURE_COLUMNS], b[FEATURE_COLUMNS])


PROVIDED_STAGE2_CHECKS = [
    check_strictly_prior_same_timestamp_and_retries,
    check_exact_window_boundary_and_first_observation,
    check_shuffled_input_does_not_change_results,
]
run_checks(PROVIDED_STAGE2_CHECKS)

### Stage 2 — your tests

Cover the cases required by the brief with small explicit fixtures.

In [ ]:
# Write your own test functions here (plain `assert`), then list them below.


CANDIDATE_STAGE2_TESTS = []
run_checks(CANDIDATE_STAGE2_TESTS)

In [ ]:
features = try_stage("Stage 2", build_online_features, transactions)
if features is not None:
    print(f"{len(features):,} logical payments")
    display(features[["transaction_id", "customer_id", "created_at", *FEATURE_COLUMNS]].head())

**Stage 2 — notes**

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 2**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 3 — Chargeback Risk Modeling

> **Brief:** released by the interviewer during the session. Paste it here when you receive it.

In [ ]:
def split_by_time(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Train / validation / test split."""
    raise NotImplementedError("Stage 3")


def train_chargeback_baseline(features_and_labels: pd.DataFrame):
    """Risk baseline, with metrics."""
    raise NotImplementedError("Stage 3")


if features is None:
    risk_result = None
    print("[Stage 3] waiting for Stage 2 features")
else:
    risk_result = try_stage("Stage 3", train_chargeback_baseline, features)

**Stage 3 — notes**

Report observed class support (n, positives) for every split.

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 3**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 4 — Decision Policy

> **Brief:** released by the interviewer during the session. Paste it here when you receive it.

In [ ]:
def choose_actions(probabilities: np.ndarray, amounts: np.ndarray,
                   max_review_rate: float = 0.015) -> np.ndarray:
    """Return APPROVE / REVIEW / BLOCK for each payment."""
    raise NotImplementedError("Stage 4")

**Stage 4 — notes**

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 4**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 5 — Payment Exceptions: Text, Embeddings and Retrieval

> **Brief:** released by the interviewer during the session. Paste it here when you receive it.

In [ ]:
def train_text_baselines(messages: pd.DataFrame, embeddings: pd.DataFrame):
    """Sparse TF-IDF and dense-embedding classifiers and comparison."""
    raise NotImplementedError("Stage 5")


def classify_with_abstention(model, X, threshold: float):
    """Class / confidence / human-fallback output."""
    raise NotImplementedError("Stage 5")


def retrieve_similar(query_vector: np.ndarray, index_vectors: np.ndarray,
                     index_ids: list[str], k: int = 3):
    """k most similar indexed examples."""
    raise NotImplementedError("Stage 5")


text_result = try_stage("Stage 5", train_text_baselines, messages, embeddings)

**Stage 5 — notes**

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 5**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 6 — Acquirer Routing

> **Brief:** released by the interviewer during the session. Paste it here when you receive it.

In [ ]:
def choose_acquirer(transaction: dict, acquirer_reference: pd.DataFrame) -> str:
    """Pick an acquirer for a new payment."""
    raise NotImplementedError("Stage 6")


example_payment = transactions.drop_duplicates("transaction_id").iloc[0].to_dict()
try_stage("Stage 6", choose_acquirer, example_payment, acquirers)

**Stage 6 — notes**

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 6**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Stage 7 — Production Architecture

> **Brief:** released by the interviewer during the session. Paste it here when you receive it.

**Stage 7 — notes**

- Approach:
- Assumptions:
- Results:
- Limitations and next steps:

**AI assistant log — Stage 7**

- What the assistant generated:
- How I verified it (tests, checks, reasoning):
- What I corrected or rejected, and why:

---
# Final Handoff

## 1. Data integrity

- Decision unit:
- Label population/observation limitation:
- Temporal split:
- Main sources of leakage avoided:

## 2. Implemented code

- Functions completed:
- Tests passed:
- Known unhandled edge cases:

## 3. Offline model metrics

| Model | Validation metrics | Test metrics | Cautions |
|---|---|---|---|
| Chargeback risk | | | |
| TF-IDF exception classifier | | | |
| 32D text-derived embedding classifier | | | |

## 4. Decision economics

- Policy objective and main constraints:
- Validation-only threshold/capacity methodology:
- Why offline policy lift cannot be assumed causal:

## 5. Proposed deployment

- Payments authorization risk + routing critical path:
- Separate payment exceptions NLP path:
- Fail-open / fail-safe / guarded-fallback assumptions:
- Monitoring, canary strategy and rollback conditions:

## 6. Your AI coding assistant

- One useful thing it accelerated:
- One mistake or questionable assumption you corrected:

In [ ]:
STAGE_FUNCTIONS = {
    "Stage 1": lambda: audit_data(transactions, labels),
    "Stage 2": lambda: build_online_features(transactions.head(50)),
    "Stage 3": lambda: split_by_time(transactions.head(50)),
    "Stage 4": lambda: choose_actions(np.array([0.1]), np.array([100.0])),
    "Stage 5": lambda: retrieve_similar(np.zeros(32), np.zeros((1, 32)), ["x"]),
    "Stage 6": lambda: choose_acquirer(transactions.iloc[0].to_dict(), acquirers),
}
status = {}
for stage, call in STAGE_FUNCTIONS.items():
    try:
        call()
        status[stage] = "implemented"
    except NotImplementedError:
        status[stage] = "not implemented"
    except Exception as exc:
        status[stage] = f"error: {type(exc).__name__}"
pd.Series(status, name="status").to_frame()